# miniLLM

A minimal, legible, and easy-to-modify decoder language model baseline. Based on Mistral 7B, it's built as a quick reference against which to test architecture and research ideas.

In [ ]:
import torch
from torch import nn, einsum
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset, Dataset
from torch.optim.lr_scheduler import _LRScheduler

import numpy as np
import math
import transformers
from transformers import AutoTokenizer
from dataclasses import dataclass
from typing import List, Optional

!pip install -q rotary-embedding-torch
from rotary_embedding_torch import RotaryEmbedding

!pip install -q datasets
import datasets
from datasets import load_dataset

# Needed if compiling torch model running on Colab
# See: https://github.com/pytorch/pytorch/issues/107960
!export LC_ALL="en_US.UTF-8"
!export LD_LIBRARY_PATH="/usr/lib64-nvidia"
!export LIBRARY_PATH="/usr/local/cuda/lib64/stubs"
!ldconfig /usr/lib64-nvidia

## Model

In [ ]:
@dataclass
class ModelArgs:
    dim: int
    n_layers: int
    head_dim: int
    hidden_dim: int
    n_heads: int
    n_kv_heads: int
    norm_eps: float
    vocab_size: int
    mla: bool


class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight


class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.n_heads: int = args.n_heads
        # Group query attention if n_kv_heads < n_heads
        self.n_kv_heads: int = args.n_kv_heads
        self.repeats = self.n_heads // self.n_kv_heads
        self.scale = self.args.head_dim**-0.5

        self.wq = nn.Linear(args.dim, args.n_heads * args.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * args.head_dim, args.dim, bias=False)

    def forward(
        self, x: torch.Tensor,
        rotary_emb_fn,
        cache = None
    ):
        batch_size, seq_len = x.shape[0], x.shape[1]

        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
        xq = xq.view(batch_size, seq_len, self.n_heads, self.args.head_dim).transpose(1,2) # -> b h t d
        xk = xk.view(batch_size, seq_len, self.n_kv_heads, self.args.head_dim).transpose(1,2)
        xv = xv.view(batch_size, seq_len, self.n_kv_heads, self.args.head_dim).transpose(1,2)


        if cache is not None:
            key_cache, value_cache = cache
            key = torch.cat([key_cache, xk], dim=1) # concatenate on seq_len
            value = torch.cat([value_cache, xv], dim=1) # concatenate on seq_len
            # lucidrains RoPE called like so for cache:
            xq, key = rotary_emb_fn.rotate_queries_with_cached_keys(xq, key)
            new_cache = (key, value)
        else:
            xq = rotary_emb_fn.rotate_queries_or_keys(xq)
            xk = rotary_emb_fn.rotate_queries_or_keys(xk)
            key, value = xk, xv
            new_cache = None

        # Repeat keys and values to match number of query heads (for GQA MQA)
        key = torch.repeat_interleave(key, repeats=self.repeats, dim=1)
        value = torch.repeat_interleave(value, repeats=self.repeats, dim=1)

        output = torch.nn.functional.scaled_dot_product_attention(
            xq, key, value,
            attn_mask = None, # If None and is_causal=True, creates causal attention mask
            dropout_p = 0.0,
            is_causal = True,
            scale = None # If None, default is 1/sqrt(dim)
        )

        output = output.transpose(1,2).view(batch_size, seq_len, self.n_heads * self.args.head_dim)

        return self.wo(output), new_cache


class MLA(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.n_heads: int = args.n_heads
        #self.head_dim = int(args.dim / args.n_heads)
        self.head_dim = args.head_dim

        # compression rank (1/4 in v3)
        self.q_rank = args.dim // 4
        self.kv_rank = args.dim // 4

        # some dimensions of k are compressed, the rest are used to create a rope head
        self.qk_nope_head_dim = self.head_dim
        self.qk_rope_head_dim = self.head_dim // 2 # v3
        # head_dim for queries and keys, values uses regular head_dim
        self.qk_head_dim = self.qk_nope_head_dim + self.qk_rope_head_dim

        # Q
        self.dq = nn.Linear(args.dim, self.q_rank, bias=False)
        self.q_norm = RMSNorm(self.q_rank, eps=args.norm_eps)
        self.uq = nn.Linear(self.q_rank, self.qk_head_dim * self.n_heads, bias=False)

        # KV
        self.dkv_nope = nn.Linear(args.dim, self.kv_rank, bias=False)
        self.dkv_norm = RMSNorm(self.kv_rank, eps=args.norm_eps)
        self.uk_rope = nn.Linear(args.dim, self.qk_rope_head_dim, bias=False)
        self.uv = nn.Linear(self.kv_rank, self.n_heads * self.head_dim, bias=False)
        self.uk_nope = nn.Linear(self.kv_rank, self.n_heads * self.qk_nope_head_dim, bias=False)

        # O
        self.wo = nn.Linear(args.n_heads * args.head_dim, args.dim, bias=False)

    def forward(
        self,
        x,
        rotary_emb_fn,
        cache = None):
        batch_size, seq_len = x.shape[0], x.shape[1]

        # KV
        kv_nope = self.dkv_nope(x)
        kv_nope = self.dkv_norm(kv_nope)
        k_rope = self.uk_rope(x)

        # Caching
        if cache is not None:
            cached_kv_nope, cached_k_rope = cache
            k_rope = torch.cat([cached_k_rope, k_rope], dim=1) # concat on seq_len
            kv_nope = torch.cat([cached_kv_nope], dim=1) # concat on seq_len
            # Cache compressed kv and k_rope
            new_cache = (kv_nope, k_rope)
        else:
            new_cache = None

        # K_nope
        k_nope = self.uk_nope(kv_nope)
        k_nope = k_nope.view(batch_size, seq_len, self.n_heads, self.qk_nope_head_dim) # b t h d_nope
        k_nope = k_nope.transpose(1,2) # b h t d_nope

        # K_rope
        k_rope = k_rope.unsqueeze(2) # b t 1 d_rope
        k_rope = k_rope.expand(-1,-1,self.n_heads,-1) # b t h d_rope
        k_rope = k_rope.transpose(1,2) # b h t d_rope

        # V
        v = self.uv(kv_nope)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim) # b t h d
        v = v.transpose(1,2) # b h t d

        q = self.uq(self.q_norm(self.dq(x)))
        q = q.view(batch_size, seq_len, self.n_heads, self.qk_head_dim) # b t h d
        q = q.transpose(1,2) # b h t d

        q_nope, q_rope = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1) # split for rope

        # Apply rope
        if cache is not None:
            q_rope, k_rope = rotary_emb_fn.rotate_queries_with_cached_keys(q_rope, k_rope)
        else:
            q_rope = rotary_emb_fn.rotate_queries_or_keys(q_rope)
            k_rope = rotary_emb_fn.rotate_queries_or_keys(k_rope)

        k_full = torch.cat([k_nope, k_rope], dim=-1) # b h t d
        q_full = torch.cat([q_nope, q_rope], dim=-1) # b h t d

        output = torch.nn.functional.scaled_dot_product_attention(
            q_full, k_full, v,
            attn_mask = None,
            dropout_p = 0.0,
            is_causal = True,
            scale = None
        )

        output = output.transpose(1,2).reshape(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.wo(output), new_cache


class FeedForward(nn.Module):
    # SwiGLU
    def __init__(self, args: ModelArgs):
        super().__init__()

        self.w1 = nn.Linear(args.dim, args.hidden_dim, bias=False)
        self.w2 = nn.Linear(args.hidden_dim, args.dim, bias=False)
        self.w3 = nn.Linear(args.dim, args.hidden_dim, bias=False)

    def forward(self, x) -> torch.Tensor:
        return self.w2(nn.functional.silu(self.w1(x)) * self.w3(x))


class TransformerBlock(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        if args.mla:
            self.attention = MLA(args)
        else:
            self.attention = Attention(args)
        self.feed_forward = FeedForward(args=args)
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.args = args

    def forward(
        self, x: torch.Tensor,
        rotary_emb_fn,
        cache = None
    ):

        h, new_cache = self.attention.forward(
            self.attention_norm(x),
            rotary_emb_fn,
            cache=cache
        )

        h = x + h
        out = h + self.feed_forward.forward(self.ffn_norm(h))

        return out, new_cache

class Transformer(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.args = args
        self.vocab_size = args.vocab_size
        self.n_layers = args.n_layers
        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)

        self.layers = torch.nn.ModuleList(
            [TransformerBlock(args=args) for _ in range(args.n_layers)]
        )

        self.norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)
        # https://github.com/lucidrains/rotary-embedding-torch
        self.rotary_emb_fn = RotaryEmbedding(dim = args.head_dim // 2)


    def forward(self, input_ids: torch.Tensor, labels=None) -> torch.Tensor:
        h = self.tok_embeddings(input_ids)

        for layer in self.layers:
            h, _ = layer(h, self.rotary_emb_fn, cache=None)

        logits = self.output(self.norm(h)).float()

        if labels is None:
            return logits

        loss = F.cross_entropy(logits.transpose(-1,-2), labels)
        return loss


    def generate(self, idx, max_new_tokens=200):

        cache = []
        for _ in range(self.n_layers):
            cache.append(None)

        for _ in range(max_new_tokens):
            if idx.size(1) > 1:
                input_pos = idx[:, -1:]
            else:
                input_pos = idx

            h = self.tok_embeddings(input_pos)

            for i, layer in enumerate(self.layers):
                h, new_cache = layer(
                    h,
                    self.rotary_emb_fn,
                    cache=cache[i]
                )
                cache[i] = new_cache

            logits = self.output(self.norm(h)).float()
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


class WarmupCosineLR(_LRScheduler):
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.1, last_epoch=-1):
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        super(WarmupCosineLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_steps:
            return [base_lr * self.last_epoch / self.warmup_steps for base_lr in self.base_lrs]
        if self.last_epoch > self.total_steps:
            return [base_lr * self.min_lr_ratio for base_lr in self.base_lrs]
        else:
            decay_ratio = (self.last_epoch - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            coeff = 0.5 * (1.0 - self.min_lr_ratio) * (1 + math.cos(math.pi * decay_ratio))
            return [base_lr * (self.min_lr_ratio + coeff) for base_lr in self.base_lrs]

## Parameters

In [ ]:
# Tells LR scheduler over what period to decay the learning rate
TOTAL_STEPS = 1e5

# Warmup steps
WARMUP_STEPS = 300

LEARNING_RATE = 1e-4
MAX_GRAD_CLIP_NORM = 1.0
BATCH_SIZE = 16
SEQ_LEN = 512 + 1

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", pad_token='[PAD]')
VOCAB_SIZE = len(tokenizer)

## Dataset

In [ ]:
class StreamingDataset(IterableDataset):
    def __init__(self, stream_dataset):
        self.stream_dataset = stream_dataset

    def __iter__(self):
        for sample in self.stream_dataset:
            text = sample['text']
            tokenized = tokenizer(text, truncation=True, padding='max_length', max_length=SEQ_LEN, return_tensors="pt")
            yield tokenized['input_ids'].squeeze(0)

dataset = load_dataset("c4", "en", streaming=True)
dataset = dataset.shuffle(seed=42, buffer_size=10_000)

train_dataset = StreamingDataset(dataset['train'])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True, drop_last=True)
valid_dataset = StreamingDataset(dataset['validation'])
valid_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True, drop_last=True)
valid_loader_iterator = iter(valid_loader)

## Initialize model

In [ ]:
# Initialize a small model
args = ModelArgs(dim=768,
                 n_layers=12,
                 head_dim=64,
                 hidden_dim=2048, # (4 * dim) for normal feedforward, (8/3 * dim) for GLU variants
                 n_heads=8,
                 n_kv_heads=4, # GQA
                 norm_eps=1e-5,
                 vocab_size=VOCAB_SIZE,
                 mla=True)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(args).to(device)
# Compile
model = torch.compile(model)
optim = torch.optim.AdamW(model.parameters(), lr = LEARNING_RATE, betas=(0.9, 0.95),  weight_decay=0.1)
# LR schedule: warmup to LR, then cosine decay to 10%
scheduler = WarmupCosineLR(optim, warmup_steps=WARMUP_STEPS, total_steps=TOTAL_STEPS)
# Automatic mixed precision with scaler
use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

## Training loop

In [ ]:
EPOCHS = 3
PRINT_LOSS = 10
GENERATE_EVERY = 100

for epoch in range(EPOCHS):
    for i, data in enumerate(train_loader):
        model.train()
        train_loss = 0.
        seq, labels = data[:, :-1], data[:, 1:]
        seq, labels = seq.to(device), labels.to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            loss = model(
                seq,
                labels = labels,
            )

        train_loss += loss.item()
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_CLIP_NORM)

        scaler.step(optim)
        scaler.update()
        optim.zero_grad(set_to_none=True)
        scheduler.step()

        if not (i % PRINT_LOSS):
            print(f'step {i} :training loss: {train_loss}')

        if not (i % GENERATE_EVERY):
            model.eval()
            for i in range(2):
                context = torch.zeros((1, 1), dtype=torch.long).to(device)
                res = model.generate(context, max_new_tokens=100)
                print ([tokenizer.decode(x) for x in res], "\n")
            model.train()

## Sample generation

In [ ]:
for i in range(3):
    context = torch.zeros((1, 1), dtype=torch.long).to(device)
    res = model.generate(context, max_new_tokens=250)
    print ([tokenizer.decode(x) for x in res], "\n")